# Colab Training Notebook for TinyStoriesZh

This notebook runs the upstream-style repo on Colab with a Google Drive-backed cache and a conservative tokenizer preparation path.

## 1. Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_URL = 'https://github.com/picasso250/autoresearch-zh.git'
BRANCH = 'codex/zh-port'
DRIVE_ROOT = Path('/content/drive/MyDrive')
WORKDIR = DRIVE_ROOT / 'colab' / 'autoresearch-zh'
REPO_DIR = WORKDIR / 'repo'
CACHE_DIR = WORKDIR / 'cache'

WORKDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)

print(f'WORKDIR:   {WORKDIR}')
print(f'REPO_DIR:  {REPO_DIR}')
print(f'CACHE_DIR: {CACHE_DIR}')

## 2. Clone or refresh the repo

In [ ]:
import subprocess

def run(cmd, cwd=None):
    print('>', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

print('Repo ready:', REPO_DIR)

## 3. Install runtime dependencies

In [ ]:
import sys
import subprocess
import torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kernels>=0.11.7',
    'pyarrow>=21.0.0',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'tiktoken>=0.11.0',
], check=True)

print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 4. Prepare repo imports

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR)
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
os.chdir(REPO_DIR)
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)

print('cwd:', Path.cwd())
print('repo import path ready:', repo_str)

## 5. Download dataset files only

In [ ]:
import prepare

DATASET = 'tinystorieszh'
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)
prepare.download_data(DATASET)
print('Dataset download step finished.')

## 6. Train tokenizer only

This uses a smaller text budget than the raw `prepare.py` default so Colab is less likely to kill the process. Adjust `--vocab-size` here when you want to test smaller tokenizer vocabularies.

In [ ]:
TOKENIZER_VOCAB_SIZE = 8192
TOKENIZER_DIR = CACHE_DIR / 'datasets' / 'tinystorieszh' / f'tokenizer-vocab-{TOKENIZER_VOCAB_SIZE}'
print('TOKENIZER_DIR:', TOKENIZER_DIR)
%cd {REPO_DIR}
!python prepare_tokenizer_t4.py --dataset tinystorieszh --vocab-size {TOKENIZER_VOCAB_SIZE} --max-chars 100000000 --doc-cap 8000

## 7. Full training run (simple T4 large-model baseline)

In [ ]:
%cd {REPO_DIR}
!python train_t4.py --dataset tinystorieszh --tokenizer-dir "{TOKENIZER_DIR}" --depth 10 --model-dim 1024 --device-batch-size 4 --lr-scale 0.125 --total-batch-size 65536

## 8. Generate from `checkpoint_last.pt`

In [ ]:
%cd {REPO_DIR}
!python generate_t4.py --dataset tinystorieszh --tokenizer-dir "{TOKENIZER_DIR}" --checkpoint checkpoint_last.pt --prompt "从前有一只小猫" --max-new-tokens 120